In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


# LPIPS and BioCLIP


## Config


In [ ]:
# companion to the FID notebook. everything here is per image or per pair,
#nothing rests on estimating a distribution from 100 samples.
#
# lpips_to_real:   distance from each generated image to held-out set
# lpips_diversity: mean pairwise distance inside a generated set, low means a collapse
# nn_train:        min distance to any training image, low = memorisation
# bioclip:         zero-shot id rate and cosine alignment with species name
#
# 

import os, shutil
from pathlib import Path

ROOT   = str(ROOT)
GEN      = os.path.join(ROOT, "outputs/generated")
BASE      = os.path.join(ROOT, "outputs/generated_baseline")
REF = os.path.join(ROOT, "data/reference")
TRAIN  = os.path.join(ROOT, "data/train")
METRICS       = os.path.join(ROOT, "outputs/metrics")
WORK      = "/content/metrics_work"

SPECIES = ["achillea", "carpobrotus", "eryngium"]
RES  = 512
N_GEN = 100

# Full binomials for the BioCLIP text side. Genus and species separately,
# 
BINOMIAL = {
    "achillea":    ("Achillea",    "maritima"),
    "carpobrotus": ("Carpobrotus", "acinaciformis"),
    "eryngium":    ("Eryngium",    "maritimum"),
}

TRAIN_FOLDERS = {
    "achillea":    ["Achillea_Maritima_2"],
    "carpobrotus": ["Carpobrotus_Acinaciformis_3"],
    "eryngium":    ["Eryngium_Maritimum_2"],
}

os.makedirs(METRICS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SKIP_FOLDERS = {"_smoketest"}

def list_images(folder):
    p = Path(folder)
    return sorted([f for f in p.rglob("*") if f.suffix.lower() in EXTS]) if p.exists() else []

def parse(folder_name, condition):
    name = folder_name.lower()
    if name in SKIP_FOLDERS:
        return None
    sp = next((s for s in SPECIES if s in name), None)
    if sp is None:
        return None
    model = name.split(sp)[0].strip("_") or "unnamed"
    return model, condition, sp


Mounted at /content/drive


In [2]:
!pip install -q lpips pybioclip open_clip_torch

import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
device: cuda


## Preprocess


In [ ]:
# same crop and resize for generated, reference and training. 
from PIL import Image

def prepare(files, dest, limit=None):
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    written = 0
    for i, f in enumerate(files if limit is None else files[:limit]):
        try:
            im = Image.open(f).convert("RGB")
        except Exception as e:
            print(f"    skipping {f.name}: {e}")
            continue
        w, h = im.size
        side = min(w, h)
        im = im.crop(((w - side) // 2, (h - side) // 2,
                      (w + side) // 2, (h + side) // 2))
        im = im.resize((RES, RES), Image.BICUBIC)
        im.save(dest / f"{i:05d}.png")
        written += 1
    return written

ref_files = {sp: list_images(Path(REF) / sp) for sp in SPECIES}

gen_files = {}
for root, cond in [(GEN, "lora"), (BASE, "base")]:
    if not os.path.exists(root):
        print(f"  MISSING: {root}")
        continue
    for folder in sorted(Path(root).iterdir()):
        if not folder.is_dir():
            continue
        parsed = parse(folder.name, cond)
        if parsed is None:
            continue
        gen_files[parsed] = list_images(folder)

print("reference")
ref_dirs = {}
for sp in SPECIES:
    d = Path(WORK) / "ref" / sp
    n = prepare(ref_files[sp], d)
    ref_dirs[sp] = str(d)
    print(f"  {sp:<14} {n:>5}")

print("\ntraining")
train_dirs = {}
for sp, folders in TRAIN_FOLDERS.items():
    files = []
    for fo in folders:
        files += list_images(Path(TRAIN) / fo)
    d = Path(WORK) / "train" / sp
    n = prepare(files, d)
    train_dirs[sp] = str(d)
    print(f"  {sp:<14} {n:>5}")

print("\ngenerated")
gen_dirs = {}
for k, files in sorted(gen_files.items()):
    model, cond, sp = k
    d = Path(WORK) / "gen" / f"{model}_{cond}_{sp}"
    n = prepare(files, d, limit=N_GEN)
    gen_dirs[k] = str(d)
    print(f"  {model:<8} {cond:<5} {sp:<14} {n:>5}")

print(f"\n{len(gen_dirs)} cells")


REFERENCE
  achillea         200
  carpobrotus      200
  eryngium         200

TRAINING
  achillea         138
  carpobrotus       70
  eryngium          90

GENERATED
  flux2    base  achillea         100
  flux2    base  carpobrotus      100
  flux2    base  eryngium         100
  flux2    lora  achillea         100
  flux2    lora  carpobrotus      100
  flux2    lora  eryngium         100
  flux3    lora  achillea         100
  flux3    lora  carpobrotus      100
  flux3    lora  eryngium         100
  pixart   base  achillea         100
  pixart   base  carpobrotus      100
  pixart   base  eryngium         100
  pixart   lora  achillea         100
  pixart   lora  carpobrotus      100
  pixart   lora  eryngium         100
  qwen     base  achillea         100
  qwen     base  carpobrotus      100
  qwen     base  eryngium         100
  qwen     lora  achillea         100
  qwen     lora  carpobrotus      100
  qwen     lora  eryngium         100
  sdxl     base  achillea        

## LPIPS


In [ ]:
# alexnet, which is what the paper recommends as a forward metric. vgg for perceptual loss. 
#
# 256px as matches alexnet was train resolution. reference capped at
# MAX_REF since fidelity is a mean over n_gen x n_ref forward passes.
import lpips
import numpy as np
import torch.nn.functional as F

LPIPS_RES = 256
MAX_REF   = 100
BATCH     = 32

loss_fn = lpips.LPIPS(net="alex").to(DEV).eval()

def load_tensor(d, limit=None):
    """Load a directory of PNGs as an N,3,H,W tensor scaled to [-1, 1]."""
    files = sorted(Path(d).glob("*.png"))
    if limit is not None:
        files = files[:limit]
    arrs = []
    for f in files:
        im = Image.open(f).convert("RGB").resize((LPIPS_RES, LPIPS_RES), Image.BICUBIC)
        arrs.append(np.asarray(im, dtype=np.float32) / 127.5 - 1.0)
    if not arrs:
        return torch.empty(0, 3, LPIPS_RES, LPIPS_RES)
    return torch.from_numpy(np.stack(arrs)).permute(0, 3, 1, 2)

@torch.no_grad()
def cross_d(a, b):
    """All pairwise LPIPS distances between tensor sets a and b. Returns 1D array."""
    out = []
    for i in range(a.shape[0]):
        ai = a[i:i+1].to(DEV)
        for s in range(0, b.shape[0], BATCH):
            chunk = b[s:s+BATCH].to(DEV)
            d = loss_fn(ai.expand(chunk.shape[0], -1, -1, -1), chunk)
            out.append(d.flatten().cpu().numpy())
    return np.concatenate(out) if out else np.array([])

@torch.no_grad()
def self_d(a):
    """Pairwise LPIPS within one set, upper triangle only."""
    out = []
    n = a.shape[0]
    for i in range(n - 1):
        ai = a[i:i+1].to(DEV)
        rest = a[i+1:]
        for s in range(0, rest.shape[0], BATCH):
            chunk = rest[s:s+BATCH].to(DEV)
            d = loss_fn(ai.expand(chunk.shape[0], -1, -1, -1), chunk)
            out.append(d.flatten().cpu().numpy())
    return np.concatenate(out) if out else np.array([])

@torch.no_grad()
def nn_d(a, b):
    """For each image in a, the minimum LPIPS distance to any image in b."""
    mins = []
    for i in range(a.shape[0]):
        ai = a[i:i+1].to(DEV)
        best = np.inf
        for s in range(0, b.shape[0], BATCH):
            chunk = b[s:s+BATCH].to(DEV)
            d = loss_fn(ai.expand(chunk.shape[0], -1, -1, -1), chunk)
            best = min(best, float(d.min()))
        mins.append(best)
    return np.array(mins)

print("lpips loaded")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 155MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
LPIPS ready


In [5]:
import pandas as pd, time

ref_tensors   = {sp: load_tensor(ref_dirs[sp], limit=MAX_REF) for sp in SPECIES}
train_tensors = {sp: load_tensor(train_dirs[sp], limit=MAX_REF) for sp in SPECIES}
for sp in SPECIES:
    print(f"  {sp:<14} ref {ref_tensors[sp].shape[0]:>4}   train {train_tensors[sp].shape[0]:>4}")

lpips_rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    t0 = time.time()
    g = load_tensor(gdir)
    if g.shape[0] == 0:
        print(f"  {model} {cond} {sp}: EMPTY, skipped")
        continue

    fid_d  = cross_d(g, ref_tensors[sp])
    div_d  = self_d(g)
    nn_d   = nn_d(g, train_tensors[sp])

    lpips_rows.append({
        "model": model, "condition": cond, "species": sp, "n_gen": int(g.shape[0]),
        "lpips_to_real":   round(float(fid_d.mean()), 4),
        "lpips_diversity": round(float(div_d.mean()), 4),
        "nn_train_mean":   round(float(nn_d.mean()), 4),
        "nn_train_min":    round(float(nn_d.min()), 4),
    })
    print(f"  {model:<8} {cond:<5} {sp:<12} "
          f"real {fid_d.mean():.3f}  div {div_d.mean():.3f}  "
          f"nn {nn_d.mean():.3f} (min {nn_d.min():.3f})   [{time.time()-t0:.0f}s]")

lpips_df = pd.DataFrame(lpips_rows)
lpips_df


  achillea       ref  100   train  100
  carpobrotus    ref  100   train   70
  eryngium       ref  100   train   90
  flux2    base  achillea     real 0.689  div 0.446  nn 0.588 (min 0.519)   [35s]
  flux2    base  carpobrotus  real 0.627  div 0.430  nn 0.520 (min 0.431)   [32s]
  flux2    base  eryngium     real 0.671  div 0.512  nn 0.599 (min 0.532)   [35s]
  flux2    lora  achillea     real 0.654  div 0.517  nn 0.483 (min 0.405)   [38s]
  flux2    lora  carpobrotus  real 0.661  div 0.621  nn 0.543 (min 0.429)   [34s]
  flux2    lora  eryngium     real 0.760  div 0.603  nn 0.548 (min 0.429)   [36s]
  flux3    lora  achillea     real 0.662  div 0.559  nn 0.491 (min 0.408)   [38s]
  flux3    lora  carpobrotus  real 0.628  div 0.581  nn 0.505 (min 0.380)   [34s]
  flux3    lora  eryngium     real 0.738  div 0.675  nn 0.538 (min 0.377)   [36s]
  pixart   base  achillea     real 0.747  div 0.601  nn 0.641 (min 0.559)   [38s]
  pixart   base  carpobrotus  real 0.737  div 0.603  nn 0.647 (

,model,condition,species,n_gen,lpips_to_real,lpips_diversity,nn_train_mean,nn_train_min
0,flux2,base,achillea,100,0.6894,0.4457,0.5880,0.5195
1,flux2,base,carpobrotus,100,0.6270,0.4295,0.5202,0.4305
2,flux2,base,eryngium,100,0.6712,0.5121,0.5992,0.5325
3,flux2,lora,achillea,100,0.6539,0.5167,0.4829,0.4046
4,flux2,lora,carpobrotus,100,0.6613,0.6213,0.5435,0.4289
5,flux2,lora,eryngium,100,0.7604,0.6032,0.5476,0.4294
6,flux3,lora,achillea,100,0.6625,0.5587,0.4907,0.4080
7,flux3,lora,carpobrotus,100,0.6283,0.5812,0.5052,0.3795
8,flux3,lora,eryngium,100,0.7377,0.6754,0.5383,0.3774
9,pixart,base,achillea,100,0.7466,0.6008,0.6414,0.5593


## Zero-shot id


In [ ]:
# TreeOfLifeClassifier over the full tree, 
# id_rate is top-1 == target species, genus_rate is top-1 in the right
# genus. a recognisable Eryngium that isn't E. maritimum fails differently
# from something unidentifiable, and the gap is where this shows.
# mean_conf is a confidence, not a probability.
from bioclip import TreeOfLifeClassifier, Rank

clf = TreeOfLifeClassifier()

def zero_shot(gdir, sp, topk=5):
    files = [str(f) for f in sorted(Path(gdir).glob("*.png"))]
    genus, epithet = BINOMIAL[sp]
    target = f"{genus} {epithet}".lower()

    hit_sp, hit_gen, confs = 0, 0, []
    for f in files:
        try:
            preds = clf.predict(f, Rank.SPECIES, k=topk)
        except Exception as e:
            print(f"    {Path(f).name}: {e}")
            continue
        if not preds:
            continue
        top = preds[0]
        name = str(top.get("species", "")).strip().lower()
        conf = float(top.get("score", 0.0))
        confs.append(conf)
        if name == target:
            hit_sp += 1
        if name.split(" ")[0] == genus.lower():
            hit_gen += 1

    n = max(len(confs), 1)
    return {
        "n_scored":   len(confs),
        "id_rate":    round(hit_sp / n, 3),
        "genus_rate": round(hit_gen / n, 3),
        "mean_conf":  round(float(np.mean(confs)) if confs else 0.0, 4),
    }

id_rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    r = zero_shot(gdir, sp)
    r.update({"model": model, "condition": cond, "species": sp})
    id_rows.append(r)
    print(f"  {model:<8} {cond:<5} {sp:<12} "
          f"id {r['id_rate']:.2f}  genus {r['genus_rate']:.2f}  conf {r['mean_conf']:.3f}")

ids_df = pd.DataFrame(id_rows)[
    ["model", "condition", "species", "n_scored", "id_rate", "genus_rate", "mean_conf"]]
ids_df


open_clip_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

embeddings/txt_emb_species.npy: reconstructing file:   0%|          |  0.00B / 2.66GB            

embeddings/txt_emb_species.npy: downloading bytes:           |  0.00B            

...

100%|██████████| 1/1 [00:02<00:00,  2.14s/images]


  sdxl     lora  carpobrotus  id 0.10  genus 1.00  conf 0.335


100%|██████████| 1/1 [00:02<00:00,  2.93s/images]

  sdxl     lora  eryngium     id 0.86  genus 0.87  conf 0.693


,model,condition,species,n_scored,id_rate,genus_rate,mean_conf
0,flux2,base,achillea,100,0.00,0.25,0.1489
1,flux2,base,carpobrotus,100,0.00,0.00,0.0992
2,flux2,base,eryngium,100,0.00,0.96,0.3085
3,flux2,lora,achillea,100,0.89,0.89,0.5682
4,flux2,lora,carpobrotus,100,0.39,0.96,0.2702
5,flux2,lora,eryngium,100,1.00,1.00,0.8898
6,flux3,lora,achillea,100,0.93,0.93,0.5837
7,flux3,lora,carpobrotus,100,0.36,0.98,0.3036
8,flux3,lora,eryngium,100,0.98,0.98,0.9051
9,pixart,base,achillea,100,0.00,0.00,0.0839


## Alignment


In [ ]:
# cosine between the BioCLIP image embedding and the species name, x100.
#Bare binomial is what TaxaDiffusion reports, so it's the comparable
# one. the photo-framed version scores higher across the board. 
import open_clip

model_bc, _, preprocess_bc = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
tokenizer_bc = open_clip.get_tokenizer("hf-hub:imageomics/bioclip-2")
model_bc = model_bc.to(DEV).eval()

PROMPTS = {
    "binomial": lambda g, e: f"{g} {e}",
    "photo":    lambda g, e: f"a photo of {g} {e}",
}

@torch.no_grad()
def text_embed(text):
    tok = tokenizer_bc([text]).to(DEV)
    t = model_bc.encode_text(tok)
    return F.normalize(t, dim=-1)

@torch.no_grad()
def image_embeds(gdir, batch=32):
    files = sorted(Path(gdir).glob("*.png"))
    outs = []
    for s in range(0, len(files), batch):
        ims = [preprocess_bc(Image.open(f).convert("RGB")) for f in files[s:s+batch]]
        x = torch.stack(ims).to(DEV)
        v = model_bc.encode_image(x)
        outs.append(F.normalize(v, dim=-1).cpu())
    return torch.cat(outs) if outs else torch.empty(0)

text_cache = {}
for sp, (g, e) in BINOMIAL.items():
    for key, fn in PROMPTS.items():
        text_cache[(sp, key)] = text_embed(fn(g, e)).cpu()

align_rows = []
for (model, cond, sp), gdir in sorted(gen_dirs.items()):
    v = image_embeds(gdir)
    if v.shape[0] == 0:
        continue
    row = {"model": model, "condition": cond, "species": sp, "n_gen": int(v.shape[0])}
    for key in PROMPTS:
        sims = (v @ text_cache[(sp, key)].T).squeeze(-1).numpy() * 100.0
        row[f"bioclip_{key}"]    = round(float(sims.mean()), 3)
        row[f"bioclip_{key}_sd"] = round(float(sims.std()), 3)
    align_rows.append(row)
    print(f"  {model:<8} {cond:<5} {sp:<12} "
          f"binomial {row['bioclip_binomial']:.2f}  photo {row['bioclip_photo']:.2f}")

align_df = pd.DataFrame(align_rows)
align_df


  flux2    base  achillea     binomial 57.61  photo 55.68
  flux2    base  carpobrotus  binomial 54.75  photo 63.41
  flux2    base  eryngium     binomial 64.65  photo 66.12
  flux2    lora  achillea     binomial 66.03  photo 74.95
  flux2    lora  carpobrotus  binomial 61.62  photo 74.34
  flux2    lora  eryngium     binomial 67.52  photo 73.98
  flux3    lora  achillea     binomial 66.26  photo 75.02
  flux3    lora  carpobrotus  binomial 62.04  photo 75.03
  flux3    lora  eryngium     binomial 68.00  photo 73.97
  pixart   base  achillea     binomial 52.51  photo 55.37
  pixart   base  carpobrotus  binomial 40.83  photo 50.86
  pixart   base  eryngium     binomial 58.13  photo 63.30
  pixart   lora  achillea     binomial 64.32  photo 73.09
  pixart   lora  carpobrotus  binomial 61.19  photo 73.43
  pixart   lora  eryngium     binomial 67.59  photo 73.71
  qwen     base  achillea     binomial 55.14  photo 54.28
  qwen     base  carpobrotus  binomial 52.18  photo 60.64
  qwen     bas

,model,condition,species,n_gen,bioclip_binomial,bioclip_binomial_sd,bioclip_photo,bioclip_photo_sd
0,flux2,base,achillea,100,57.613,2.606,55.680,2.324
1,flux2,base,carpobrotus,100,54.751,2.257,63.413,1.669
2,flux2,base,eryngium,100,64.646,1.688,66.124,1.681
3,flux2,lora,achillea,100,66.029,1.344,74.954,1.260
4,flux2,lora,carpobrotus,100,61.616,1.447,74.344,1.113
5,flux2,lora,eryngium,100,67.518,0.956,73.978,0.573
6,flux3,lora,achillea,100,66.263,1.071,75.019,0.947
7,flux3,lora,carpobrotus,100,62.043,1.293,75.032,1.149
8,flux3,lora,eryngium,100,68.000,0.776,73.974,0.550
9,pixart,base,achillea,100,52.512,3.956,55.366,3.039


## Real-image ceilings


In [ ]:
# every metric above, run on the held-out real photos. Necessary for interpreting the numbers.
# the diversity ceiling is important as a set well below it has collapsed even if fidelity looks acceptable
ceil_rows = []
for sp in SPECIES:
    r = ref_tensors[sp]
    t = train_tensors[sp]

    real_to_real = cross_d(r, t)          # held-out vs training, both real
    real_div     = self_d(r)              # spread within real held-out
    ids          = zero_shot(ref_dirs[sp], sp)

    v = image_embeds(ref_dirs[sp])
    align = {}
    for key in PROMPTS:
        sims = (v @ text_cache[(sp, key)].T).squeeze(-1).numpy() * 100.0
        align[f"bioclip_{key}"] = round(float(sims.mean()), 3)

    row = {
        "species": sp,
        "lpips_to_real":   round(float(real_to_real.mean()), 4),
        "lpips_diversity": round(float(real_div.mean()), 4),
        **{k: ids[k] for k in ["id_rate", "genus_rate", "mean_conf"]},
        **align,
    }
    ceil_rows.append(row)
    print(f"  {sp:<14} lpips_real {row['lpips_to_real']:.3f}  "
          f"div {row['lpips_diversity']:.3f}  id {row['id_rate']:.2f}  "
          f"bioclip {row['bioclip_binomial']:.2f}")

ceilings = pd.DataFrame(ceil_rows)
ceilings


100%|██████████| 1/1 [00:02<00:00,  2.94s/images]


  achillea       lpips_real 0.665  div 0.647  id 0.89  bioclip 66.85


100%|██████████| 1/1 [00:02<00:00,  2.95s/images]


  carpobrotus    lpips_real 0.641  div 0.598  id 0.23  bioclip 63.67


100%|██████████| 1/1 [00:02<00:00,  2.57s/images]


  eryngium       lpips_real 0.651  div 0.615  id 0.97  bioclip 68.69


,species,lpips_to_real,lpips_diversity,id_rate,genus_rate,mean_conf,bioclip_binomial,bioclip_photo
0,achillea,0.6650,0.6467,0.885,0.885,0.5851,66.847,75.209
1,carpobrotus,0.6407,0.5976,0.225,0.980,0.3647,63.671,74.773
2,eryngium,0.6511,0.6149,0.970,0.970,0.9119,68.693,74.209


In [11]:
ref_rows = []
for sp, files in ref_files.items():
    v = image_embeds(Path(REF) / sp)
    if v.shape[0] == 0:
        continue
    row = {"species": sp, "n": int(v.shape[0])}
    sims = (v @ text_cache[(sp, "photo")].T).squeeze(-1).numpy() * 100.0
    row["bioclip_photo"] = round(float(sims.mean()), 3)
    row["bioclip_photo_sd"] = round(float(sims.std()), 3)
    for other in ref_files:
        if other != sp:
            s = (v @ text_cache[(other, "photo")].T).squeeze(-1).numpy() * 100.0
            row[f"vs_{other}"] = round(float(s.mean()), 2)
    ref_rows.append(row)

pd.DataFrame(ref_rows)


,species,n,bioclip_photo,bioclip_photo_sd,vs_carpobrotus,vs_eryngium,vs_achillea
0,achillea,200,75.209,1.141,59.09,63.26,NaN
1,carpobrotus,200,74.773,1.186,NaN,59.39,58.89
2,eryngium,200,74.209,0.782,60.75,NaN,65.79


## Export


In [9]:
keys = ["model", "condition", "species"]
per_image = (lpips_df
             .merge(ids_df.drop(columns=["n_scored"]), on=keys, how="outer")
             .merge(align_df.drop(columns=["n_gen"]), on=keys, how="outer"))

by_arm = (per_image
          .groupby(["model", "condition"])
          .agg(lpips_to_real=("lpips_to_real", "mean"),
               lpips_diversity=("lpips_diversity", "mean"),
               nn_train_mean=("nn_train_mean", "mean"),
               id_rate=("id_rate", "mean"),
               genus_rate=("genus_rate", "mean"),
               bioclip=("bioclip_binomial", "mean"),
               n=("id_rate", "size"))
          .round(4)
          .sort_values("id_rate", ascending=False))

per_image.to_csv(os.path.join(METRICS, "per_image_metrics.csv"), index=False)
by_arm.to_csv(os.path.join(METRICS, "per_image_by_arm.csv"))
ceilings.to_csv(os.path.join(METRICS, "per_image_ceilings.csv"), index=False)

print("wrote to", METRICS)
by_arm


Written to /content/drive/MyDrive/Synthetic_Plants_Project/Data_Clean


,,lpips_to_real,lpips_diversity,nn_train_mean,id_rate,genus_rate,bioclip,n
model,condition,,,,,,,
flux2,lora,0.6919,0.5804,0.5247,0.7600,0.9500,65.0543,3
qwen,lora,0.6660,0.5489,0.5036,0.7600,0.9133,64.7930,3
flux3,lora,0.6762,0.6051,0.5114,0.7567,0.9633,65.4353,3
sdxl,lora,0.6342,0.5164,0.4920,0.6233,0.9267,65.6830,3
pixart,lora,0.6114,0.5633,0.4951,0.5067,0.7600,64.3687,3
flux2,base,0.6625,0.4624,0.5691,0.0000,0.4033,59.0033,3
pixart,base,0.7505,0.6207,0.6570,0.0000,0.0133,50.4907,3
qwen,base,0.7472,0.4623,0.6526,0.0000,0.4400,57.2077,3
sdxl,base,0.6727,0.5828,0.5834,0.0000,0.3367,60.9913,3


## LaTeX table


In [ ]:
# Source of table used in the paper. 
sub = per_image[per_image["condition"] == "lora"]
agg = (sub.groupby("model")
       .agg(lpips=("lpips_to_real", "mean"),
            div=("lpips_diversity", "mean"),
            idr=("id_rate", "mean"),
            bc=("bioclip_binomial", "mean"))
       .round(3)
       .sort_values("idr", ascending=False))

ceil = ceilings.mean(numeric_only=True)

lines = [
    r"\begin{table}[htbp]", r"\centering", r"\footnotesize",
    r"\setlength{\tabcolsep}{4pt}",
    r"\caption{Automated per-image metrics, LoRA condition, averaged across the "
    r"three species. LPIPS is measured against held-out real photographs (lower "
    r"is closer); diversity is mean pairwise LPIPS within each generated set "
    r"(higher is more varied); ID rate is the proportion of images assigned to "
    r"the target species by BioCLIP-2; alignment is the BioCLIP-2 image-text "
    r"cosine similarity $\times 100$. The final row gives the values obtained "
    r"by real photographs under the same pipeline.}",
    r"\label{tab:per-image-metrics}",
    r"\begin{tabular}{lcccc}", r"\toprule",
    r"Model & LPIPS $\downarrow$ & Diversity & ID rate $\uparrow$ & Alignment $\uparrow$ \\",
    r"\midrule",
]
for model, r in agg.iterrows():
    lines.append(f"{model} & {r['lpips']:.3f} & {r['div']:.3f} & "
                 f"{r['idr']:.3f} & {r['bc']:.2f} \\\\")
lines += [
    r"\midrule",
    (f"Real photographs & {ceil['lpips_to_real']:.3f} & {ceil['lpips_diversity']:.3f} & "
     f"{ceil['id_rate']:.3f} & {ceil['bioclip_binomial']:.2f} \\\\"),
    r"\bottomrule", r"\end{tabular}", r"\end{table}",
]

table_tex = "\n".join(lines)
with open(os.path.join(METRICS, "per_image_table.tex"), "w") as f:
    f.write(table_tex)
print(table_tex)


\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Automated per-image metrics, LoRA condition, averaged across the three species. LPIPS is measured against held-out real photographs (lower is closer); diversity is mean pairwise LPIPS within each generated set (higher is more varied); ID rate is the proportion of images assigned to the target species by BioCLIP-2; alignment is the BioCLIP-2 image-text cosine similarity $\times 100$. The final row gives the values obtained by real photographs under the same pipeline.}
\label{tab:per-image-metrics}
\begin{tabular}{lcccc}
\toprule
Model & LPIPS $\downarrow$ & Diversity & ID rate $\uparrow$ & Alignment $\uparrow$ \\
\midrule
flux2 & 0.692 & 0.580 & 0.760 & 65.05 \\
qwen & 0.666 & 0.549 & 0.760 & 64.79 \\
flux3 & 0.676 & 0.605 & 0.757 & 65.44 \\
sdxl & 0.634 & 0.516 & 0.623 & 65.68 \\
pixart & 0.611 & 0.563 & 0.507 & 64.37 \\
\midrule
Real photographs & 0.652 & 0.620 & 0.693 & 66.40 \\
\bottomrule
\end{tabular